In [17]:
# %%
import salvus.namespace as sn
import sympy as sp
import numpy as np
import numpy.typing as npt


# %% [markdown]
# Let's create a simple 4 element mesh.

# %%
mesh = sn.simple_mesh.CartesianHomogeneousAcoustic2D(
    vp=1, rho=1, x_max=1, y_max=1, max_frequency=1
).create_mesh()

# Change the polynomial order to 7, so we can have some complex functions
# inside the elements
mesh.change_tensor_order(4)

mesh

In [18]:
mesh.npoint

81

In [24]:
points = mesh.points
points.T.shape

def v(x, y):
    # sine with period 0.5 in both directions
    return np.sin(2 * np.pi * x)


nodal_velocity = v(*points.T)
nodal_velocity.shape

(81,)

In [27]:
mesh.connectivity

array([[ 0,  9, 18, 27, 36,  1, 10, 19, 28, 37,  2, 11, 20, 29, 38,  3,
        12, 21, 30, 39,  4, 13, 22, 31, 40],
       [36, 45, 54, 63, 72, 37, 46, 55, 64, 73, 38, 47, 56, 65, 74, 39,
        48, 57, 66, 75, 40, 49, 58, 67, 76],
       [ 4, 13, 22, 31, 40,  5, 14, 23, 32, 41,  6, 15, 24, 33, 42,  7,
        16, 25, 34, 43,  8, 17, 26, 35, 44],
       [40, 49, 58, 67, 76, 41, 50, 59, 68, 77, 42, 51, 60, 69, 78, 43,
        52, 61, 70, 79, 44, 53, 62, 71, 80]])

In [30]:
elemental_nodal_velocity = nodal_velocity[mesh.connectivity]
elemental_nodal_velocity

array([[ 0.00000000e+00,  5.16251887e-01,  1.00000000e+00,
         5.16251887e-01,  1.22464680e-16,  0.00000000e+00,
         5.16251887e-01,  1.00000000e+00,  5.16251887e-01,
         1.22464680e-16,  0.00000000e+00,  5.16251887e-01,
         1.00000000e+00,  5.16251887e-01,  1.22464680e-16,
         0.00000000e+00,  5.16251887e-01,  1.00000000e+00,
         5.16251887e-01,  1.22464680e-16,  0.00000000e+00,
         5.16251887e-01,  1.00000000e+00,  5.16251887e-01,
         1.22464680e-16],
       [ 1.22464680e-16, -5.16251887e-01, -1.00000000e+00,
        -5.16251887e-01, -2.44929360e-16,  1.22464680e-16,
        -5.16251887e-01, -1.00000000e+00, -5.16251887e-01,
        -2.44929360e-16,  1.22464680e-16, -5.16251887e-01,
        -1.00000000e+00, -5.16251887e-01, -2.44929360e-16,
         1.22464680e-16, -5.16251887e-01, -1.00000000e+00,
        -5.16251887e-01, -2.44929360e-16,  1.22464680e-16,
        -5.16251887e-01, -1.00000000e+00, -5.16251887e-01,
        -2.44929360e-16],
    

In [31]:
# We actually don't need any pre-existing fields
mesh.elemental_fields.clear()

# Attach the ENF to the mesh for visualization
mesh.attach_field("VP", elemental_nodal_velocity)

# %%
mesh

In [ ]:
# ---
# jupyter:
#   jupytext:
#     formats: ipynb,py:percent
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.17.3
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# %% [markdown]
# # Averaging Elemental Nodal Fields (ENF) to Elemental Fields (EF)
# ## Lars Gebraad, Mondaic AG, 2025-10-09
#
# This notebook demonstrates how to correctly average an elemental nodal field to
# an elemental (cell-wise constant) field.
#
# It does two averages:
# 1. Naive (unweighted) arithmetic mean over the nodal values in each element,
#    and show how that is insufficient.
# 2. Physically meaningful average using the spectral element mass matrix
#    (i.e., quadrature-weighted integration over the element).
#
# The mass matrix provided by Salvus (via `mesh.get_mass_matrix()`) already
# contains the product \\( w_i J_i \\) per local node, enabling a weighted average.

# %%
import salvus.namespace as sn
import sympy as sp
import numpy as np
import numpy.typing as npt


# %% [markdown]
# Let's create a simple 4 element mesh.

# %%
mesh = sn.simple_mesh.CartesianHomogeneousAcoustic2D(
    vp=1, rho=1, x_max=1, y_max=1, max_frequency=1
).create_mesh()

# Change the polynomial order to 7, so we can have some complex functions
# inside the elements
mesh.change_tensor_order(7)

mesh
# %% [markdown]
# Let's extract the points s.t. we can attach a complicated field.
# %%
points = mesh.points


# %% [markdown]
# We define a velocity field that varies in `x` only, with period 0.5. We then
# evaluate it at the nodal points and reshape to elemental nodal field (ENF).
# %%
def v(x, y):
    # sine with period 0.5 in both directions
    return np.sin(2 * np.pi * x)


nodal_velocity = v(*points.T)
elemental_nodal_velocity = nodal_velocity[mesh.connectivity]

# We actually don't need any pre-existing fields
mesh.elemental_fields.clear()

# Attach the ENF to the mesh for visualization
mesh.attach_field("VP", elemental_nodal_velocity)

# %%
mesh

# %% [markdown]
# ## Naive arithmetic mean
# The simplest way to reduce an elemental nodal field (shape: `(nelem,
# nodes_per_element)`) to an elemental field (shape: `(nelem,)`) is to take the
# plain arithmetic mean over the nodal axis. This implicitly assumes:
# - Uniform spacing of interpolation points,
# - Unit (or equal) quadrature weights,
# - No geometric distortion (Jacobian constant and factored out).
#
# These assumptions are generally violated for anything but first order
# non-deformed elements, so the naive mean is typically pretty inaccurate, even
# useless.

# %%
naive_average = elemental_nodal_velocity.mean(axis=-1)
print("Naive average:", naive_average)

# %% [markdown]
# ## Analytical reference (first element)
# For the first element we can actually compute the average per element also
# analytically. The first element spans `[0, 0.5] x [0, 0.5]` in `(x, y)`, so
# the average velocity is:
#
# $$
# \overline{\text{\;vp\;}} = \frac{\int _0^{0.5} \int_0^{0.5} \sin(2 \pi x) \, dy \, dx}{\int_0^{0.5} \int_0^{0.5} 1 \, dy \, dx}
# $$

# %%
x = sp.Symbol("x")
v_sp = sp.sin(2 * sp.pi * x)

# Integral of velocity along x axis, times length along y axis
element_length = 1 / (mesh.nelem**0.5)
area = element_length**2
average_analytical = (
    sp.integrate(v_sp, (x, 0, element_length)) * element_length
)
average_analytical /= area

print("Analytical average (first element):", float(average_analytical))

# %% [markdown]
# That's nowhere near the naive average:
# %%
print(
    f"Element 0 -> naive: {naive_average[0]:.6f}, "
    f"analytical: {float(average_analytical):.6f}"
)

# %% [markdown]
# ## Mass-matrix-weighted averaging
# The correct SEM average uses quadrature (GLL) weights and the geometric
# mapping. Salvus can easily give you the mass matrix entries per element node
# through `mesh.get_mass_matrix()`.
#
# By multiplying the elemental nodal field with the mass matrix and summing
# over the nodes, we get a quadrature-weighted integral over the element, i.e.
# the numerator of the average formula above. Dividing by the sum of the mass
# matrix entries (which is the element volume) gives the average value.

# %% [markdown]
# Let's first have a look at the mass matrix:
# %%
mm = mesh.get_mass_matrix()

mesh.elemental_fields.clear()
mesh.attach_field("mass_matrix", mm)
mesh
# %% [markdown]
# The shapes of the mass matrix is just an elemental nodal field.
# %%
print(
    "Mass matrix shape:",
    mm.shape,
    "| ENF shape:",
    elemental_nodal_velocity.shape,
)

# %% [markdown]
# Now we can compute the weighted average:
# %%
VP_elemental = (mm * elemental_nodal_velocity).sum(axis=-1) / mm.sum(axis=-1)

print("SEM average:", VP_elemental)
print("Analytical average (first element):", float(average_analytical))
# %%


def average_enf_to_ef(
    enf: npt.NDArray, mesh: sn.UnstructuredMesh
) -> npt.NDArray:
    """
    Average an elemental nodal field to an elemental field for a mesh.

    Args:
        enf: Elemental nodal field of shape (nelem, nodes_per_element).
        mesh: The mesh to use for the averaging.

    Returns
        ef: Elemental field of shape (nelem,).
    """
    mm = mesh.get_mass_matrix()
    ef = (mm * enf).sum(axis=-1) / mm.sum(axis=-1)
    return ef


average_enf_to_ef(elemental_nodal_velocity, mesh)
